### Building one neural network and pointing it at four problems

This notebook defines a small feed-forward network once, then trains it on four
different targets without changing a single thing about its structure. Holding
the machinery fixed is the entire point: whatever differs in the results comes
from the problem, not from the network.

The four problems, in order of how well they turn out:

| Problem | Input | Target |
| --- | --- | --- |
| XOR | 2 bits | 1 bit, is exactly one input a 1 |
| Population count | 8 bits | 4 bits, how many 1 bits are set |
| Primality | 8 bits | 4 bits, a code word for prime or composite |
| Collatz stopping time | 8 bits | 7 bits, steps needed to reach 1 |

The last three each hold back a handful of integers the network never sees.
Scoring on data it trained on tells you only that it **memorized**. Scoring on
the integers held back tells you whether it **learned** a rule that transfers.
The final problem is expected to fail, and that failure is the lesson.

In [ ]:
"""neural_network.ipynb"""

# Cell 01 - Import packages and define the activation functions

%matplotlib inline

import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rich
from IPython.display import display, update_display
from rich import box
from rich.panel import Panel
from rich.table import Table
from rich.theme import Theme

# Every table in this notebook prints in one plain color, with no cell
# emphasized over another. Nothing in how a number looks should suggest
# what it means: the numbers are compared against each other, and the
# markdown and the verdict panels say what the comparison shows.
PLAIN = Theme(
    {
        "table.title": "none",
        "table.header": "none",
        "table.footer": "none",
        "table.caption": "none",
        "rule.line": "none",
        "rule.text": "none",
    }
)

# Reconfigure Rich's own console rather than building a separate one, so
# that the tables handed to display() in Cells 10, 13 and 16 render the
# same way as the ones printed here. Fixing the width keeps the rules and
# panels the same size on any screen, and highlight=False stops Rich from
# coloring numbers on its own
rich.reconfigure(width=88, highlight=False, theme=PLAIN)
console = rich.get_console()


def relu(z: np.ndarray) -> np.ndarray:
    """Pass positive values through unchanged and clamp negatives to zero."""
    return np.maximum(0, z)


def relu_derivative(a: np.ndarray) -> np.ndarray:
    """Derivative of relu, given a = relu(z) rather than z."""
    return (a > 0).astype(float)


def sigmoid(z: np.ndarray) -> np.ndarray:
    """Logistic sigmoid, evaluated so that exp() cannot overflow.

    The direct form 1 / (1 + exp(-z)) overflows once z is about -710,
    because exp(-z) then exceeds the largest float64. For negative z the
    algebraically identical form exp(z) / (1 + exp(z)) is used instead,
    where the exponent stays negative and exp(z) never exceeds 1.
    """
    z = np.asarray(z, dtype=float)
    result = np.empty_like(z)

    positive = z >= 0
    result[positive] = 1 / (1 + np.exp(-z[positive]))

    exp_z = np.exp(z[~positive])
    result[~positive] = exp_z / (1 + exp_z)

    return result


def sigmoid_derivative(a: np.ndarray) -> np.ndarray:
    """Derivative of the sigmoid, given a = sigmoid(z) rather than z."""
    return a * (1 - a)


probe = np.array([-800.0, -1.0, 0.0, 1.0, 800.0])
print(f"relu    {relu(probe)}")
print(f"sigmoid {sigmoid(probe)}")
print("The plain 1 / (1 + exp(-z)) form overflows at z = -800")

---
### The network

Five layers of neurons: an input layer, three hidden layers of equal width, and
an output layer. The hidden layers use `relu`, the output layer uses `sigmoid`
so that every output lands in $(0, 1)$ and can be rounded to a bit.

Every layer computes $w \cdot x + b$. The bias $b$ shifts the point at which a
neuron turns on. Without it a neuron could only respond to $w \cdot x$, whose
dividing line is pinned to the origin, and an all-zeros input would produce the
same answer no matter how long the network trained.

The two initialization schemes are chosen to match the activation that follows
them. **He** initialization scales by $\sqrt{2 / n_{in}}$ for the relu layers,
**Xavier** by $\sqrt{1 / n_{in}}$ for the sigmoid output. Biases start at zero:
unlike the weights they do not need random values to break symmetry, because
the incoming weights already differ neuron to neuron.

In [ ]:
# Cell 02 - The network, with three hidden layers and a bias on every neuron


class SimpleNeuralNetwork:
    def __init__(self, input_size: int, hidden_size: int, output_size: int):
        self.input_size = input_size
        self.hidden_size = hidden_size  # Each hidden layer has hidden_size neurons
        self.output_size = output_size

        # He initialization for the layers with relu activation
        self.weights_input_hidden1 = np.random.randn(input_size, hidden_size) * np.sqrt(
            2.0 / input_size
        )
        self.weights_hidden1_hidden2 = np.random.randn(
            hidden_size, hidden_size
        ) * np.sqrt(2.0 / hidden_size)
        self.weights_hidden2_hidden3 = np.random.randn(
            hidden_size, hidden_size
        ) * np.sqrt(2.0 / hidden_size)

        # Xavier initialization for the output layer with sigmoid activation
        self.weights_hidden3_output = np.random.randn(
            hidden_size, output_size
        ) * np.sqrt(1.0 / hidden_size)

        # One bias per neuron, stored as a row so it broadcasts across
        # every sample in the batch
        self.bias_hidden1 = np.zeros((1, hidden_size))
        self.bias_hidden2 = np.zeros((1, hidden_size))
        self.bias_hidden3 = np.zeros((1, hidden_size))
        self.bias_output = np.zeros((1, output_size))

    def parameter_count(self) -> int:
        """Return how many weights and biases the network holds."""
        return sum(
            array.size
            for array in (
                self.weights_input_hidden1,
                self.weights_hidden1_hidden2,
                self.weights_hidden2_hidden3,
                self.weights_hidden3_output,
                self.bias_hidden1,
                self.bias_hidden2,
                self.bias_hidden3,
                self.bias_output,
            )
        )

    def topology(self) -> str:
        """Return the layer sizes as a printable string."""
        hidden = self.hidden_size
        return f"{self.input_size} -> {hidden} -> {hidden} -> {hidden} -> {self.output_size}"

    def forward(self, x: np.ndarray) -> np.ndarray:
        # Each layer computes w . x + b, then applies its activation
        self.hidden1_input = np.dot(x, self.weights_input_hidden1) + self.bias_hidden1
        self.hidden1_output = relu(self.hidden1_input)

        self.hidden2_input = (
            np.dot(self.hidden1_output, self.weights_hidden1_hidden2)
            + self.bias_hidden2
        )
        self.hidden2_output = relu(self.hidden2_input)

        self.hidden3_input = (
            np.dot(self.hidden2_output, self.weights_hidden2_hidden3)
            + self.bias_hidden3
        )
        self.hidden3_output = relu(self.hidden3_input)

        self.final_input = (
            np.dot(self.hidden3_output, self.weights_hidden3_output) + self.bias_output
        )
        self.final_output = sigmoid(self.final_input)

        return self.final_output

    def backward(
        self, x: np.ndarray, y: np.ndarray, output: np.ndarray, learning_rate: float
    ) -> None:
        # Output error and delta, using the sigmoid derivative
        self.loss = y - output
        self.output_delta = self.loss * sigmoid_derivative(output)

        # Push the error back through the three relu layers
        hidden3_error = self.output_delta.dot(self.weights_hidden3_output.T)
        hidden3_delta = hidden3_error * relu_derivative(self.hidden3_output)

        hidden2_error = hidden3_delta.dot(self.weights_hidden2_hidden3.T)
        hidden2_delta = hidden2_error * relu_derivative(self.hidden2_output)

        hidden1_error = hidden2_delta.dot(self.weights_hidden1_hidden2.T)
        hidden1_delta = hidden1_error * relu_derivative(self.hidden1_output)

        # Update the weights, each scaled by the activation feeding into it
        self.weights_hidden3_output += learning_rate * self.hidden3_output.T.dot(
            self.output_delta
        )
        self.weights_hidden2_hidden3 += learning_rate * self.hidden2_output.T.dot(
            hidden3_delta
        )
        self.weights_hidden1_hidden2 += learning_rate * self.hidden1_output.T.dot(
            hidden2_delta
        )
        self.weights_input_hidden1 += learning_rate * x.T.dot(hidden1_delta)

        # Update the biases. A bias has no incoming activation to scale it,
        # so its gradient is just the delta summed over the batch
        self.bias_output += learning_rate * np.sum(
            self.output_delta, axis=0, keepdims=True
        )
        self.bias_hidden3 += learning_rate * np.sum(
            hidden3_delta, axis=0, keepdims=True
        )
        self.bias_hidden2 += learning_rate * np.sum(
            hidden2_delta, axis=0, keepdims=True
        )
        self.bias_hidden1 += learning_rate * np.sum(
            hidden1_delta, axis=0, keepdims=True
        )

    def train(
        self,
        x: np.ndarray,
        y: np.ndarray,
        epochs: int = 10_000,
        learning_rate: float = 0.005,
        report_every: int = 2_000,
    ) -> np.ndarray:
        """Run gradient descent, returning the error recorded at every epoch.

        Pass report_every=0 to train silently, which is what the
        cross-validation in Cell 11 does across its 16 runs.
        """
        history = np.zeros(epochs)
        for epoch in range(epochs):
            output = self.forward(x)
            self.backward(x, y, output, learning_rate)
            history[epoch] = np.mean(np.abs(self.loss))
            if report_every and epoch % report_every == 0:
                print(f"  epoch {epoch:>6,}   error {history[epoch]:.5f}")
        if report_every:
            print(f"  epoch {epochs:>6,}   error {history[-1]:.5f}")
        return history


probe_network = SimpleNeuralNetwork(input_size=8, hidden_size=16, output_size=4)
print(
    f"An {probe_network.topology()} network holds "
    f"{probe_network.parameter_count():,} parameters"
)
print("  (expected 756 = 704 weights + 52 biases)")

---
### Splitting the data, and saving the trained network

`choose_held_out` picks the 16 integers that no network in this notebook will
ever train on, and `split_for` converts that one list into array positions for
whichever problem is running. Every problem therefore holds back the **same 16
integers**, which is what lets the results be compared directly.

How those 16 are chosen matters more than it might appear, because a held-out
set that was picked with any knowledge of the answers would quietly decide the
result before a single epoch ran. Three properties keep it clean:

- **The draw sees only the integers.** It is handed the range 1 to 255 and
  nothing else. It never evaluates a population count, a primality test, or a
  stopping time, so no property of any answer can steer which integers are held
  back. Every integer in the range is equally likely.
- **The draw happens before any training.** The 16 come out first, are removed
  from the data, and no network ever sees them. There is no path for a held-out
  answer to leak into training.
- **The seed fixes which draw you get, not how it is drawn.** Seeding makes the
  run reproducible so a result can be checked and argued with. Change
  `SPLIT_SEED` and you get a different 16, drawn exactly as blindly.

The one integer excluded is 0, and that is structural rather than a judgment
about difficulty: the Collatz sequence is undefined for 0, so 0 cannot belong to
a set shared with that problem.

Sixteen is a small sample, and for two of the three problems that turns out not
to matter, because the results are not close. For primality it matters a great
deal, and Cell 11 shows what to do about it.

`save_model` writes the weights, the biases, and the layer sizes to a compressed
`.npz` file, along with any extra arrays passed to it. The four problems use that
to record which integers were trained on, which were held back, and the training
curve, so a later cell can rebuild everything from the file alone.

`load_network` reads a file back and returns a network already sized to match it,
which means a report can never disagree with the run that produced it.

In [ ]:
# Cell 03 - Choose the held-out integers, then save and reload a network

ARCH_KEYS = ("input_size", "hidden_size", "output_size")
WEIGHT_KEYS = (
    "weights_input_hidden1",
    "weights_hidden1_hidden2",
    "weights_hidden2_hidden3",
    "weights_hidden3_output",
    "bias_hidden1",
    "bias_hidden2",
    "bias_hidden3",
    "bias_output",
)


HELD_OUT_COUNT, SPLIT_SEED = 16, 2024


def choose_held_out(count: int, seed: int) -> np.ndarray:
    """Pick the integers that no network in this notebook will train on.

    The draw is blind, and staying blind is the point. It looks at the
    integers 1 through 255 and nothing else. It never consults a
    population count, a primality test, or a stopping time, so no
    property of any answer can influence which integers get held back.
    Every integer in the range is equally likely to be drawn.

    Drawing without replacement means no integer is held back twice, and
    seeding the generator makes the same 16 come up on every run, so a
    result can be reproduced and argued about. A seed fixes which draw
    you get; it does not make the draw any less blind.

    0 is the one integer excluded, and that is a structural constraint
    rather than a judgment about difficulty: the Collatz sequence is
    undefined for 0, so 0 cannot belong to a set shared with that
    problem.
    """
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(np.arange(1, 256), size=count, replace=False))


HELD_OUT = choose_held_out(HELD_OUT_COUNT, SPLIT_SEED)


def split_for(integers: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return array positions for the training and held-out samples.

    Each problem stores its samples in its own array, so the same integer
    sits at a different position in each. This converts the shared list of
    held-out integers into positions valid for the array given.
    """
    held = np.isin(integers, HELD_OUT)
    return np.flatnonzero(~held), np.flatnonzero(held)


def save_model(network: SimpleNeuralNetwork, filename: Path, **extra) -> None:
    """Save the parameters, the layer sizes, and any extra arrays given."""
    np.savez_compressed(
        filename,
        input_size=network.input_size,
        hidden_size=network.hidden_size,
        output_size=network.output_size,
        **{key: getattr(network, key) for key in WEIGHT_KEYS},
        **extra,
    )
    print(f"Saved {filename.name}")


def load_network(filename: Path) -> tuple[SimpleNeuralNetwork, dict]:
    """Rebuild a saved network, sized to match the file, plus its extras."""
    data = np.load(filename)
    network = SimpleNeuralNetwork(
        input_size=int(data["input_size"]),
        hidden_size=int(data["hidden_size"]),
        output_size=int(data["output_size"]),
    )
    for key in WEIGHT_KEYS:
        setattr(network, key, data[key])
    print(f"Loaded {filename.name}")
    return network, {k: data[k] for k in data.files if k not in ARCH_KEYS + WEIGHT_KEYS}


# Notebooks have no __file__, so anchor the .npz files to the working directory
LAB_DIR = Path.cwd()

print(f"Held out from every problem: {[int(n) for n in HELD_OUT]}")
print(f"Trained on: the other {255 - HELD_OUT_COUNT} integers, plus 0 where defined")
print(f"Files will be written to {LAB_DIR}")

---
### The report format

Every problem below prints the same five blocks in the same order, so the four
outputs can be read straight across:

1. **TITLE** naming the problem
2. **SETUP** the network, the data, and how the answer is encoded
3. **SCORES** one row per group of samples, plus a baseline that does no learning
4. **DETAIL** the individual answers, where there are few enough to list
5. **VERDICT** one sentence saying what the scores mean

The score table always reports a baseline. A percentage on its own is misleading
whenever one answer is far more common than the others, and for primality it is
badly misleading: answering "composite" every single time already scores 75
percent on the held-out set.

The verdict is judged two different ways, because the four problems ask for two
different kinds of answer. When the target is a **label**, `verdict_vs_baseline`
compares accuracy. When it is a **magnitude**, `verdict_vs_error` compares the
average size of the error instead, because counting exact hits is the wrong test
there: landing on 1 of 16 answers exactly while missing the rest by 30 steps is
not an improvement over answering the same number every time.

**How to read a score table.** Every row is printed the same way, so nothing
about a row's appearance tells you whether it is good news. Read the numbers
instead, always as a comparison:

- Find the **held out** row and the **baseline** row directly beneath it. The
  baseline answers without looking at the input at all, so it is the score to
  beat, not zero.
- If the held-out score is far above the baseline, the network found a rule that
  transfers to samples it never saw.
- If the two are close, the network learned nothing that generalizes, no matter
  how healthy the percentage looks by itself.
- If the held-out score is far below the **trained on** row, the network
  memorized its training data rather than learning the rule behind it.

The **verdict** panel underneath states which of those happened, in words. It
draws on one of three readings, worded identically for every problem so the four
reports can be compared sentence by sentence:

- *the network learned the rule, and it works on samples it never saw*
- *no better than the baseline, so the network learned nothing*
- *worse than not training at all, so the network learned nothing*

"Learned" always carries the meaning set out at the top of this notebook: the
network found a rule that transfers to data it was never shown. A network that
scores well on its training data and nowhere else has only memorized.

In [ ]:
# Cell 04 - Shared formatting for the four problem reports

BAR_WIDTH = 34


def print_title(problem: str, task: str) -> None:
    """Draw a rule naming the problem and what it is learning."""
    console.rule(f"{problem}  -  {task}")


def print_setup(network: str, parameters: int, data: str, target: str) -> None:
    """Show the four lines that describe the run, as a borderless table."""
    table = Table(box=None, show_header=False, pad_edge=False)
    table.add_column()
    table.add_column()
    table.add_row("Network", network)
    table.add_row("Parameters", f"{parameters:,}  (weights + biases)")
    table.add_row("Data", data)
    table.add_row("Target", target)
    console.print(table)


def print_scores(rows: list[tuple], show_error: bool = False) -> None:
    """Show the score table.

    Each row is (label, correct, total, average_error, note). Pass a total
    of None for a group that does not exist, such as the held-out set of a
    problem that trains on everything. Its row is still shown, with dashes,
    so the table shape never changes.
    """
    table = Table(title="Scores", box=box.SIMPLE_HEAVY, title_justify="left")
    table.add_column("")
    table.add_column("correct", justify="right")
    table.add_column("accuracy", justify="right")
    if show_error:
        table.add_column("avg error", justify="right")
    table.add_column("")

    for label, correct, total, average_error, note in rows:
        if total:
            cells = [label, f"{correct}/{total}", f"{100 * correct / total:.1f}%"]
        else:
            cells = [label, "-", "-"]
        if show_error:
            cells.append("-" if average_error is None else f"{average_error:.2f}")
        cells.append(note)

        table.add_row(*cells)

    console.print(table)


def print_detail(title: str, headers: list[str], rows: list[list]) -> None:
    """Show a small table of individual answers under a heading."""
    if not rows:
        console.print(f"{title}\n  (none)")
        return

    table = Table(title=title, box=box.SIMPLE, title_justify="left")
    for header in headers:
        table.add_column(header, justify="right")
    for row in rows:
        table.add_row(*[str(value) for value in row])
    console.print(table)


def print_verdict(text: str) -> None:
    """Show the closing reading of the scores. Rich wraps it to the panel."""
    console.print(Panel(text, title="Verdict", title_align="left"))


def progress_view(label: str, done: int, total: int, started: float) -> Table:
    """Build the status line shown while a slow cell is working.

    A cell that trains 16 networks takes long enough that a student needs
    to see it is alive. Displaying this with a display_id and updating
    that same id replaces the line in place rather than printing a new
    one per step, so the cell ends up holding a single finished status
    rather than a wall of progress messages.
    """
    elapsed = time.perf_counter() - started
    remaining = elapsed / done * (total - done) if done else None

    # The bar is drawn from characters rather than color, so it reads the
    # same as every other table in the notebook
    filled = round(BAR_WIDTH * done / total)
    bar = "[" + "=" * filled + "-" * (BAR_WIDTH - filled) + "]"

    table = Table(box=box.SIMPLE, title=label, title_justify="left")
    table.add_column("progress")
    table.add_column("done", justify="right")
    table.add_column("elapsed", justify="right")
    table.add_column("remaining", justify="right")
    table.add_row(
        bar,
        f"{done}/{total}",
        f"{elapsed:.0f}s",
        "-" if remaining is None else f"{remaining:.0f}s",
    )
    return table


# The three possible readings, worded once and shared by both verdict
# functions, so every problem in the notebook is judged in the same terms:
# did the network learn the rule, or only memorize what it was shown?
LEARNED = "the network learned the rule, and it works on samples it never saw"
LEARNED_NOTHING = "no better than the baseline, so the network learned nothing"
WORSE_THAN_NOTHING = "worse than not training at all, so the network learned nothing"


def verdict_vs_baseline(held_out: float, baseline: float, extra: str = "") -> str:
    """Phrase a held-out accuracy against a baseline that does no learning.

    Use this when the answer is a label, where being wrong is simply
    being wrong. For a magnitude, use verdict_vs_error instead.
    """
    lift = held_out - baseline
    if lift > 0.05:
        reading = LEARNED
    elif lift < -0.05:
        reading = WORSE_THAN_NOTHING
    else:
        reading = LEARNED_NOTHING

    sentence = f"held out {held_out:.1%} against a {baseline:.1%} baseline"
    if extra:
        sentence += f" ({extra})"
    return f"{sentence}\n{reading}"


def verdict_vs_error(held_error: float, baseline_error: float, extra: str = "") -> str:
    """Phrase an average error against a baseline that does no learning.

    When the answer is a magnitude, counting exact hits is the wrong test.
    Landing on 1 of 16 answers exactly while missing the other 15 by 30
    steps is not better than answering the same number every time, but an
    exact-match rate of 1/16 against 0/16 would score it as an improvement.
    Comparing the average error asks the question that actually matters:
    are the answers closer?
    """
    if held_error < 0.75 * baseline_error:
        reading = LEARNED
    elif held_error > 1.25 * baseline_error:
        reading = WORSE_THAN_NOTHING
    else:
        reading = LEARNED_NOTHING

    sentence = f"off by {held_error:.2f} on average against {baseline_error:.2f}"
    if extra:
        sentence += f" ({extra})"
    return f"{sentence}\n{reading}"


def decode_bits(rows: np.ndarray, output_bits: int) -> np.ndarray:
    """Round each row of sigmoid outputs and read it back as an integer."""
    place_value = 2 ** np.arange(output_bits - 1, -1, -1)
    return (np.round(rows) * place_value).sum(axis=1).astype(int)


def score_row(label, predicted, wanted, note=""):
    """Build one row of the score table from two arrays of answers.

    An average error only means something when the answer is a magnitude.
    For a yes/no answer it is just 1 minus the accuracy, which the table
    already shows, so it is left out.
    """
    predicted, wanted = np.asarray(predicted), np.asarray(wanted)
    magnitude = predicted.dtype != bool and wanted.dtype != bool
    return (
        label,
        int((predicted == wanted).sum()),
        len(wanted),
        float(np.abs(predicted - wanted).mean()) if magnitude else None,
        note,
    )


def to_bits(value: int, width: int) -> list[int]:
    """Return value as a list of bits, most significant first."""
    return [int(bit) for bit in format(value, f"0{width}b")]


def plot_history(history: np.ndarray, title: str) -> None:
    """Plot the training error against the epoch number."""
    plt.figure(figsize=(8, 4))
    plt.plot(history, color="tab:blue")
    plt.title(f"{title}\nTraining Error vs. Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Mean absolute error")
    plt.yscale("log")
    plt.grid(which="both", alpha=0.4)
    plt.show()


# Quick check that the shared blocks render the way the problems expect
print_title("example", "how every report below is laid out")
print_setup("8 -> 16 -> 16 -> 16 -> 4", 756, "256 integers, 240 / 16", "4 bits")
print_scores(
    [("held out", 15, 16, 0.06, ""), ("baseline", 7, 16, 0.88, "answer 4")],
    show_error=True,
)
print_verdict(verdict_vs_error(0.06, 0.88, "15/16 exact against 7/16"))

---
### Settings shared by the three 8-bit problems, and cross-validation

The three 8-bit problems use identical settings, so the only thing that differs
between them is the target function. Only XOR sets its own, and it says why.

`cross_validate` is the tool each problem reaches for after its report. A single
run holds back 16 integers, which is enough to illustrate but not to conclude:
one integer either way swings that score by 6.2 points. Cross-validation cuts
all the integers into folds and trains one network per fold, each predicting only
the fold it never saw. Every integer ends up tested exactly once, so the result
rests on all of them rather than on 16, and nothing had to be chosen.

It costs 16 trainings, which is why it shows a progress bar as it goes.

Each cross-validation table puts the trained network on the first row and the
baselines that learn nothing underneath it. Compare the `avg error` column down
the rows: a network worth having sits well below every baseline, and one that
sits level with them or above has gained nothing from all that training.

In [ ]:
# Cell 05 - Settings shared by the 8-bit problems, and the k-fold routine

HIDDEN_SIZE, EPOCHS, LEARNING_RATE = 16, 10_000, 0.0025
CV_FOLDS = 16


def cross_validate(
    x: np.ndarray, y: np.ndarray, output_bits: int, label: str
) -> np.ndarray:
    """Predict every sample using a network that never saw that sample.

    The samples are shuffled and cut into CV_FOLDS folds. Each fold is
    held out in turn while a fresh network trains on all the others, then
    that network predicts only its own fold. Every sample is therefore
    tested exactly once, by a network for which it was unseen data.

    The shuffle is as blind as the draw in Cell 03: it permutes positions
    and never looks at a target value.

    Training this many networks takes long enough to look like a hang, so
    the fold counter is shown and updated as the work proceeds.
    """
    order = np.random.default_rng(SPLIT_SEED).permutation(len(x))
    predicted = np.zeros(len(x), dtype=int)

    started = time.perf_counter()
    display(progress_view(label, 0, CV_FOLDS, started), display_id=label)

    for k, fold in enumerate(np.array_split(order, CV_FOLDS)):
        fold_train = np.setdiff1d(order, fold)
        np.random.seed(SPLIT_SEED + k)
        fold_nn = SimpleNeuralNetwork(
            input_size=8, hidden_size=HIDDEN_SIZE, output_size=output_bits
        )
        fold_nn.train(
            x[fold_train],
            y[fold_train],
            epochs=EPOCHS,
            learning_rate=LEARNING_RATE,
            report_every=0,
        )
        predicted[fold] = decode_bits(fold_nn.forward(x[fold]), output_bits)

        # Replaces the status in place rather than printing another line
        update_display(progress_view(label, k + 1, CV_FOLDS, started), display_id=label)

    return predicted


def print_cv_scores(rows: list[tuple]) -> None:
    """Show what a cross-validated run and its baselines achieved.

    Each row is (label, exact, total, average_error, note), the same
    shape print_scores uses, so the two tables read the same way. The
    first row is the network and the rest are baselines that learn
    nothing, so the reading is the gap between the rows.
    """
    table = Table(
        title=f"Cross-validated over every sample, {CV_FOLDS} folds",
        box=box.SIMPLE_HEAVY,
        title_justify="left",
    )
    table.add_column("")
    table.add_column("exact", justify="right")
    table.add_column("accuracy", justify="right")
    table.add_column("avg error", justify="right")
    table.add_column("")
    for label, exact, total, average_error, note in rows:
        table.add_row(
            label,
            f"{exact}/{total}",
            f"{100 * exact / total:.1f}%",
            f"{average_error:.2f}",
            note,
        )
    console.print(table)


print(
    f"Every 8-bit problem uses 8 -> {HIDDEN_SIZE} -> {HIDDEN_SIZE} -> "
    f"{HIDDEN_SIZE} -> (output), {EPOCHS:,} epochs, learning rate {LEARNING_RATE}"
)
print(f"Cross-validation trains {CV_FOLDS} networks per problem")

---
### Problem 1: XOR

XOR is the classic first test of a hidden layer. No single straight line
separates the two rows that should output 1 from the two that should output 0,
so a network with no hidden layer cannot do it at all.

The network here is 2-8-8-8-1, far more than XOR needs. Two things force the
size. The class always builds three hidden layers, and given three relu layers a
width of 4 is unreliable: `relu` is flat at zero for negative input, so a neuron
pushed negative on all four training rows receives no gradient and can never
recover. With only 4 rows an entire layer of 4 goes dark often enough to fail
the demonstration, while width 8 does not.

All four rows are the whole truth table, so nothing is held back. There is no
fifth case to generalize to.

In [ ]:
# Cell 06 - Train the network on the four rows of the XOR truth table

XOR_HIDDEN, XOR_EPOCHS, XOR_RATE = 8, 10_000, 0.1

# The XOR truth table: the output is 1 when exactly one input is 1
xor_x = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
xor_y = np.array([[0], [1], [1], [0]])

# Fix the random weights so every student sees the same run
np.random.seed(2024)
xor_nn = SimpleNeuralNetwork(input_size=2, hidden_size=XOR_HIDDEN, output_size=1)

print(f"Training {xor_nn.topology()}, {xor_nn.parameter_count():,} parameters")
xor_history = xor_nn.train(xor_x, xor_y, epochs=XOR_EPOCHS, learning_rate=XOR_RATE)

---
### Reading the XOR result

The held-out row is printed with dashes rather than left out, so this table keeps
exactly the shape used by the three problems that follow. The absence of a
held-out set is a fact about XOR worth seeing, not something to hide.

In [ ]:
# Cell 07 - Report the XOR result and plot its training curve

xor_output = xor_nn.forward(xor_x)
xor_predicted = np.round(xor_output[:, 0]).astype(int)
xor_wanted = xor_y[:, 0]

print_title("XOR", "learn the XOR truth table")
print_setup(
    network=xor_nn.topology(),
    parameters=xor_nn.parameter_count(),
    data=f"4 rows of the truth table, all {len(xor_x)} trained on, 0 held back",
    target="1 bit: 1 when exactly one input is 1",
)
print_scores(
    [
        ("trained on", int((xor_predicted == xor_wanted).sum()), len(xor_x), None, ""),
        ("held out", 0, None, None, "the truth table has no unseen rows"),
        ("baseline", int((xor_wanted == 0).sum()), len(xor_x), None, "always answer 0"),
    ]
)
print_detail(
    "Every row of the truth table",
    ["x", "actual", "predicted", "output"],
    [
        [
            f"{xor_x[i, 0]} {xor_x[i, 1]}",
            xor_wanted[i],
            xor_predicted[i],
            f"{xor_output[i, 0]:.4f}",
        ]
        for i in range(len(xor_x))
    ],
)
print_verdict(
    "nothing was held back, and nothing could be\n"
    "all 4 rows are the whole truth table, so memorizing them is a complete solution"
)

plot_history(xor_history, "XOR (2-8-8-8-1)")

---
### Problem 2: population count

From here on the input is an 8-bit integer, so 256 cases exist and some can be
held back. That is what makes the difference between memorizing and learning
measurable for the first time.

Population count is the number of 1 bits in an integer, which runs 0 to 8 and so
needs 4 output bits. It is a **smooth** function of the input bits: flip any
single bit and the answer moves by exactly 1. That smoothness is exactly what a
network can generalize from.

The size of the training set and `LEARNING_RATE` are not independent here. The
network adds up one weight update per training sample rather than averaging
them, so the real step size is `LEARNING_RATE` times the number of training
samples. Halve the training set and the network takes half-sized steps, which
looks like a learning result but is not. Change one and scale the other to
match.

In [ ]:
# Cell 08 - Train on population count, then save the network to an .npz file


def popcount(n: int) -> int:
    """Return how many 1 bits are set in n."""
    total = 0
    while n > 0:
        total += n % 2
        n //= 2
    return total


# 8-bit input rows, shared by all three of the 8-bit problems
integers_8bit = np.arange(256)
x_8bit = np.array([to_bits(int(n), 8) for n in integers_8bit])

pop_actual = np.array([popcount(int(n)) for n in integers_8bit])
pop_y = np.array([to_bits(int(v), 4) for v in pop_actual])

pop_train, pop_test = split_for(integers_8bit)

np.random.seed(2024)
pop_nn = SimpleNeuralNetwork(input_size=8, hidden_size=HIDDEN_SIZE, output_size=4)
print(f"Training {pop_nn.topology()}, {pop_nn.parameter_count():,} parameters")
pop_history = pop_nn.train(
    x_8bit[pop_train], pop_y[pop_train], epochs=EPOCHS, learning_rate=LEARNING_RATE
)

save_model(
    pop_nn,
    LAB_DIR / "nn_popcount_weights.npz",
    train_idx=pop_train,
    test_idx=pop_test,
    history=pop_history,
)

---
### Reading the population count result

Everything below is rebuilt from the `.npz` file, not from the variables above.
That is the same path a separate script would take, and it proves the saved file
carries everything needed to reproduce the report.

This problem reports an **average error** column, because the target is a
magnitude rather than a label: being off by 1 is different from being off by 40.

In [ ]:
# Cell 09 - Reload the saved network and report the population count result

network, saved = load_network(LAB_DIR / "nn_popcount_weights.npz")
train_idx, test_idx = saved["train_idx"], saved["test_idx"]

predicted_train = decode_bits(network.forward(x_8bit[train_idx]), 4)
predicted_test = decode_bits(network.forward(x_8bit[test_idx]), 4)

# The baseline does no learning at all: it answers the most common
# population count in the training set, every single time
common = int(np.bincount(pop_actual[train_idx]).argmax())

print_title("Population count", "count the 1 bits in an 8-bit integer")
print_setup(
    network=network.topology(),
    parameters=network.parameter_count(),
    data=f"256 integers (0..255), {len(train_idx)} trained on, "
    f"{len(test_idx)} held back",
    target="4 bits: the population count, an integer from 0 to 8",
)

rows = [
    score_row("trained on", predicted_train, pop_actual[train_idx]),
    score_row("held out", predicted_test, pop_actual[test_idx]),
    score_row(
        "baseline",
        np.full(len(test_idx), common),
        pop_actual[test_idx],
        f"always answer {common}",
    ),
]
print_scores(rows, show_error=True)

print_detail(
    "Held-out mistakes",
    ["n", "actual", "predicted", "error"],
    [
        [n, pop_actual[n], p, f"{p - pop_actual[n]:+d}"]
        for n, p in sorted(zip(test_idx, predicted_test))
        if p != pop_actual[n]
    ],
)

held_out, baseline = rows[1], rows[2]
print_verdict(
    verdict_vs_error(
        held_out[3],
        baseline[3],
        f"{held_out[1]}/{held_out[2]} exact against {baseline[1]}/{baseline[2]}",
    )
)

plot_history(saved["history"], "Population count (8-16-16-16-4)")

---
### Cross-validating population count

The report above rests on 16 integers. This scores all 256, each one predicted by
a network that never saw it, so nothing rides on which 16 happened to be drawn.

The baseline is the same do-nothing rule the report used: answer the most common
population count every time.

In [ ]:
# Cell 10 - Cross-validate population count across all 256 integers

cv_pop = cross_validate(x_8bit, pop_y, 4, "Population count")

pop_common = int(np.bincount(pop_actual).argmax())
pop_constant = np.full(len(pop_actual), pop_common)

# Keep these for Cell 17, which puts all three problems on one scale
cv_pop_error = float(np.abs(cv_pop - pop_actual).mean())
cv_pop_baseline = float(np.abs(pop_constant - pop_actual).mean())

print_cv_scores(
    [
        (
            "the trained network",
            int((cv_pop == pop_actual).sum()),
            len(pop_actual),
            cv_pop_error,
            "",
        ),
        (
            "always answer " + str(pop_common),
            int((pop_constant == pop_actual).sum()),
            len(pop_actual),
            cv_pop_baseline,
            "learns nothing",
        ),
    ]
)

---
### Problem 3: primality

Identical network, identical training budget, identical split. Only the target
changes: 9 (binary `1001`) for prime and 6 (binary `0110`) for composite. The two
code words are bitwise complements, so they differ in all 4 bits.

Watch the baseline row, which matters more here than anywhere else. Only 54 of
the 256 integers are prime, so answering "composite" every single time already
scores about 80 percent. An overall percentage flatters a network that has
simply learned to say "composite".

That is why the score table breaks the held-out row into its two classes. The
held-out composites are nearly free, in the sense that the do-nothing baseline
gets every one of them. The held-out primes are the entire question, and a blind
draw of 16 integers contains only 2 or 3 of them. Read this report as an
illustration, not a conclusion; the next cell is what actually settles it.

The target is a yes/no label rather than a magnitude, so this report drops the
average error column. For a boolean answer the average error is just 1 minus the
accuracy, which the table already shows.

In [ ]:
# Cell 11 - Train on primality, then save the network to an .npz file


def is_prime(n: int) -> bool:
    """Return True if n is prime."""
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    return all(n % factor != 0 for factor in range(3, int(np.sqrt(n)) + 1, 2))


PRIME_CODE, COMPOSITE_CODE = [1, 0, 0, 1], [0, 1, 1, 0]  # 9 and 6

prime_actual = np.array([is_prime(int(n)) for n in integers_8bit])
prime_y = np.array([PRIME_CODE if p else COMPOSITE_CODE for p in prime_actual])

prime_train, prime_test = split_for(integers_8bit)

np.random.seed(2024)
prime_nn = SimpleNeuralNetwork(input_size=8, hidden_size=HIDDEN_SIZE, output_size=4)
print(f"{prime_actual.sum()} primes and {(~prime_actual).sum()} composites in 0..255")
print(f"Training {prime_nn.topology()}, {prime_nn.parameter_count():,} parameters")
prime_history = prime_nn.train(
    x_8bit[prime_train],
    prime_y[prime_train],
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
)

save_model(
    prime_nn,
    LAB_DIR / "nn_primes_weights.npz",
    train_idx=prime_train,
    test_idx=prime_test,
    history=prime_history,
)

---
### Reading the primality result

Compare the held-out row against the baseline row directly underneath it. If the
two are level, the network found nothing that transfers, no matter how healthy
the percentage looks on its own.

In [ ]:
# Cell 12 - Reload the saved network and report the primality result

network, saved = load_network(LAB_DIR / "nn_primes_weights.npz")
train_idx, test_idx = saved["train_idx"], saved["test_idx"]


def decode_primality(rows: np.ndarray) -> np.ndarray:
    """Read each row of sigmoid outputs back as True for prime."""
    return decode_bits(rows, 4) == 9


predicted_train = decode_primality(network.forward(x_8bit[train_idx]))
predicted_test = decode_primality(network.forward(x_8bit[test_idx]))

print_title("Primality", "decide whether an 8-bit integer is prime")
print_setup(
    network=network.topology(),
    parameters=network.parameter_count(),
    data=f"256 integers (0..255), {len(train_idx)} trained on, "
    f"{len(test_idx)} held back",
    target="4 bits: 1001 (9) for prime, 0110 (6) for composite",
)

held_truth = prime_actual[test_idx]
rows = [
    score_row("trained on", predicted_train, prime_actual[train_idx]),
    score_row("held out", predicted_test, held_truth),
    score_row(
        "baseline",
        np.zeros(len(test_idx), dtype=bool),
        held_truth,
        "always answer composite",
    ),
    # Split the held-out score by class. An overall percentage hides the
    # only interesting question: did it recognize any of the primes?
    score_row(
        "  of the primes",
        predicted_test[held_truth],
        held_truth[held_truth],
        "these are what the network has to find",
    ),
    score_row(
        "  of the composites",
        predicted_test[~held_truth],
        held_truth[~held_truth],
        "the baseline already gets all of these",
    ),
]
print_scores(rows)

print_detail(
    "Held-out mistakes",
    ["n", "actual", "predicted"],
    [
        [n, prime_actual[n], p]
        for n, p in sorted(zip(test_idx, predicted_test))
        if p != prime_actual[n]
    ],
)

# This report deliberately does not reach a verdict. One integer either
# way moves the held-out score by 6.2 points, which is larger than any
# difference worth reading here
held_out, baseline = rows[1], rows[2]
print_verdict(
    f"held out {held_out[1] / held_out[2]:.1%} against a "
    f"{baseline[1] / baseline[2]:.1%} baseline, finding "
    f"{int(predicted_test[held_truth].sum())} of the {int(held_truth.sum())} "
    f"held-out primes\n"
    f"far too few primes to settle anything - Cell 11 scores all "
    f"{int(prime_actual.sum())} of them"
)

plot_history(saved["history"], "Primality (8-16-16-16-4)")

---
### Settling primality properly, with cross-validation

Sixteen held-out integers cannot answer this question. Only 2 or 3 of them are
prime, so each one is worth a third or more of the score, and a different
`SPLIT_SEED` swings the result wildly. That is not a flaw in the draw; it is
what a sample of 16 can tell you.

**Cross-validation** removes both the small sample and the question of which
integers were held back. Cut all 256 into 16 folds, then train 16 networks, each
one holding out a different fold. Every integer is now predicted by a network
that never saw it, and every integer gets tested exactly once. Nothing is
selected, and all 54 primes are scored instead of 2.

The two rule-based predictors in the table are the real test. Neither learns
anything; each just reads the input bits directly. The first uses one bit, the
second adds a divisibility check. Compare the trained network against them: a
network that cannot beat arithmetic this simple has not discovered anything about
primality, whatever its accuracy says.

---
### Scoring a problem where one answer is far more common

Only 54 of the 256 integers are prime, and that lopsidedness breaks the obvious
way of keeping score. Answering "composite" every single time is right 202 times
out of 256, which is **78.9 percent accuracy** for a rule that never looks at its
input. Any honest measure has to see through that.

Start by sorting every prediction into four buckets:

| | network says prime | network says composite |
| --- | --- | --- |
| **actually prime** | true positive, `TP` | false negative, `FN` |
| **actually composite** | false positive, `FP` | true negative, `TN` |

Plain accuracy is $(TP + TN) / 256$. With 202 composites, `TN` dominates the sum
and drowns out everything the two prime columns say.

**Balanced accuracy** fixes that by scoring each class separately and averaging:

$$\text{balanced accuracy} = \frac{1}{2}\left(\frac{TP}{TP + FN} + \frac{TN}{TN + FP}\right)$$

The first fraction is the share of primes found, the second the share of
composites correctly left alone. Answering "composite" every time now scores
exactly 50 percent, which is what a do-nothing rule deserves.

**Matthews correlation coefficient** is stricter still:

$$\text{MCC} = \frac{TP \times TN - FP \times FN}{\sqrt{(TP + FP)(TP + FN)(TN + FP)(TN + FN)}}$$

It is not an accuracy at all. It is the ordinary correlation coefficient between
the predicted answers and the true ones, with prime written as $+1$ and composite
as $-1$. That gives it the range of any correlation: $+1.0$ for perfect
agreement, $0.0$ for a predictor no better than chance, and negative for one that
is reliably backwards.

**Why report both.** Balanced accuracy still misses something, and the table
below shows exactly what. The rule "odd means prime" finds 53 of the 54 primes
and scores a healthy 80.5 percent balanced accuracy, but it reaches that by
calling all 128 odd integers prime. Only 53 of those 128 guesses are right, so
when it says "prime" it is wrong 59 percent of the time. Balanced accuracy never
looks at that ratio; MCC does, through the $TP + FP$ term in its denominator, and
marks the rule down to $+0.498$.

So read the MCC column as the summary and the others as detail. A predictor earns
a high MCC only by getting all four buckets right at once.

In [ ]:
# Cell 13 - Cross-validate primality across all 256 integers

cv_predicted = cross_validate(x_8bit, prime_y, 4, "Primality") == 9


def prime_scores(predicted: np.ndarray) -> tuple[int, int, float, float]:
    """Score a primality predictor over all 256 integers.

    Returns the primes it found, the false alarms it raised, its balanced
    accuracy, and its Matthews correlation coefficient.
    """
    true_pos = int((predicted & prime_actual).sum())
    false_neg = int((~predicted & prime_actual).sum())
    false_pos = int((predicted & ~prime_actual).sum())
    true_neg = int((~predicted & ~prime_actual).sum())

    balanced = (
        true_pos / (true_pos + false_neg) + true_neg / (true_neg + false_pos)
    ) / 2
    spread = np.sqrt(
        float(
            (true_pos + false_pos)
            * (true_pos + false_neg)
            * (true_neg + false_pos)
            * (true_neg + false_neg)
        )
    )
    mcc = (true_pos * true_neg - false_pos * false_neg) / spread if spread else 0.0
    return true_pos, false_pos, balanced, mcc


table = Table(
    title=f"Every integer scored by a network that never saw it ({CV_FOLDS} folds)",
    box=box.SIMPLE_HEAVY,
    title_justify="left",
)
table.add_column("predictor")
table.add_column("primes found", justify="right")
table.add_column("false alarms", justify="right")
table.add_column("balanced acc", justify="right")
table.add_column("MCC", justify="right")

candidates = [
    ("always answer composite", np.zeros(256, dtype=bool)),
    ("odd means prime (1 bit)", integers_8bit % 2 == 1),
    (
        "odd and not a multiple of 3",
        (integers_8bit % 2 == 1) & (integers_8bit % 3 != 0),
    ),
    ("the trained network", cv_predicted),
]
for name, predicted in candidates:
    found, alarms, balanced, mcc = prime_scores(predicted)
    table.add_row(
        name,
        f"{found}/{int(prime_actual.sum())}",
        str(alarms),
        f"{balanced:.1%}",
        f"{mcc:+.3f}",
    )

console.print(table)
console.print("50.0% balanced accuracy and 0.000 MCC both mean no better than chance")

# Cell 17 reuses these to place primality alongside the other two problems
_, _, cv_balanced, cv_mcc = prime_scores(cv_predicted)

---
### What that table is actually saying

Both rule-based predictors beat the trained network. Read down the MCC column,
which is the one that weighs all four buckets at once, and the network comes
last.

Do not read the other columns as a ranking on their own. "Odd means prime" raises
more false alarms than the network does, 75 against 37, which looks worse until
you notice it also finds 53 of the 54 primes where the network finds 22. It is
casting a far wider net, and catching far more with it. MCC settles that trade
and puts the rule at +0.498 against the network's +0.217.

The strongest predictor in the table is "odd and not a multiple of 3" at +0.693.
Adding a single extra divisibility check gives up one prime and removes 42 of the
75 false alarms. Two lines of arithmetic, no training of any kind, and more than
three times the network's MCC.

The network did not learn primality. It learned one bit. Every even integer
except 2 is composite, and that fact is sitting in bit 0 of the input where it is
easy to pick up and correct for 127 of the 128 even integers. Nearly all of the
network's score is that single rule. On the odd integers, where parity gives no
help and something like real primality testing would be needed, it is at chance.

So read that MCC not as a weak grasp of primality but as a smudged copy of a rule
that was visible in the input all along, and nothing beyond it. The result is
stronger than "the network did poorly". It is that the network found nothing a
single `n % 2 == 0` does not already give you, and it applies that rule less
reliably than simply reading the bit would.

Two things follow, and together they are why this problem is in the notebook at
all. First, a network can train to 99 percent on data it has seen while having
learned essentially nothing. Second, the only way to discover that is to test it
on data it has not seen, against a baseline that does no learning at all.

---
### Problem 4: Collatz stopping time

This one is expected to fail, and that is the point.

Population count is smooth in the input bits: flip any single bit and the answer
moves by exactly 1. Collatz stopping time is not. Flip one bit and the answer
moves by 33 on average, and by as much as 121. Neighboring integers are unrelated:

    n = 26  ->   10 steps
    n = 27  ->  111 steps

There is no rule connecting the bit pattern to the stopping time for the network
to find, so nothing carries over to an integer it has not seen. It can still
memorize the training set almost perfectly, which is exactly what makes the
held-out score the honest measure.

Two differences from the earlier problems, both forced by the target rather than
chosen. Stopping times below 256 reach 127, so this needs **7 output bits** where
the others needed 4. And the sequence is undefined for 0, so this problem starts
at 1 and has 255 samples instead of 256. Neither difference is why it fails: the
extra output bits give it slightly **more** capacity than the popcount network,
and it still fails completely.

In [ ]:
# Cell 14 - Train on Collatz stopping time, then save the network

COLLATZ_BITS, FIRST_N, LAST_N = 7, 1, 255


def stop_time(n: int) -> int:
    """Return how many Collatz steps it takes for n to reach 1."""
    steps = 0
    while n > 1:
        n = n // 2 if n % 2 == 0 else 3 * n + 1
        steps += 1
    return steps


collatz_integers = np.arange(FIRST_N, LAST_N + 1)
collatz_x = np.array([to_bits(int(n), 8) for n in collatz_integers])
collatz_actual = np.array([stop_time(int(n)) for n in collatz_integers])
collatz_y = np.array([to_bits(int(v), COLLATZ_BITS) for v in collatz_actual])

collatz_train, collatz_test = split_for(collatz_integers)

np.random.seed(2024)
collatz_nn = SimpleNeuralNetwork(
    input_size=8, hidden_size=HIDDEN_SIZE, output_size=COLLATZ_BITS
)
print(f"n = 26 -> {stop_time(26)} steps, but n = 27 -> {stop_time(27)} steps")
print(f"Training {collatz_nn.topology()}, {collatz_nn.parameter_count():,} parameters")
collatz_history = collatz_nn.train(
    collatz_x[collatz_train],
    collatz_y[collatz_train],
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
)

save_model(
    collatz_nn,
    LAB_DIR / "nn_collatz_weights.npz",
    train_idx=collatz_train,
    test_idx=collatz_test,
    history=collatz_history,
)

---
### Reading the Collatz result

Because a stopping time is a magnitude, this report compares against **two**
baselines that do no learning at all: always answering the median of the training
stopping times, and copying the answer of the closest training integer in bit
space. If the network cannot beat those, it has found no usable rule.

Every held-out integer is listed rather than just the mistakes, because the size
of the errors is the lesson here.

In [ ]:
# Cell 15 - Reload the saved network and report the Collatz result

network, saved = load_network(LAB_DIR / "nn_collatz_weights.npz")
train_idx, test_idx = saved["train_idx"], saved["test_idx"]

predicted_train = decode_bits(network.forward(collatz_x[train_idx]), COLLATZ_BITS)
predicted_test = decode_bits(network.forward(collatz_x[test_idx]), COLLATZ_BITS)


def nearest_neighbor_answers() -> np.ndarray:
    """Copy the answer of the training integer differing in the fewest bits."""
    answers = []
    for i in test_idx:
        distance = [
            (int(collatz_integers[i]) ^ int(collatz_integers[j])).bit_count()
            for j in train_idx
        ]
        answers.append(collatz_actual[train_idx[int(np.argmin(distance))]])
    return np.array(answers)


median = int(np.median(collatz_actual[train_idx]))

print_title("Collatz stopping time", "predict how many steps an integer needs")
print_setup(
    network=network.topology(),
    parameters=network.parameter_count(),
    data=f"255 integers (1..255), {len(train_idx)} trained on, "
    f"{len(test_idx)} held back",
    target="7 bits: the stopping time, an integer from 0 to 127",
)

rows = [
    score_row("trained on", predicted_train, collatz_actual[train_idx]),
    score_row("held out", predicted_test, collatz_actual[test_idx]),
    score_row(
        "baseline",
        np.full(len(test_idx), median),
        collatz_actual[test_idx],
        f"always answer {median}",
    ),
    score_row(
        "baseline",
        nearest_neighbor_answers(),
        collatz_actual[test_idx],
        "copy the nearest neighbor",
    ),
]
print_scores(rows, show_error=True)

print_detail(
    "Every held-out integer",
    ["n", "actual", "predicted", "error"],
    [
        [n, a, p, f"{p - a:+d}"]
        for n, a, p in sorted(
            zip(collatz_integers[test_idx], collatz_actual[test_idx], predicted_test)
        )
    ],
)

held_out, baseline = rows[1], rows[2]
print_verdict(
    verdict_vs_error(
        held_out[3],
        baseline[3],
        f"{held_out[1]}/{held_out[2]} exact against {baseline[1]}/{baseline[2]}",
    )
)

plot_history(saved["history"], "Collatz stopping time (8-16-16-16-7)")

---
### Cross-validating Collatz stopping time

Sixteen held-out integers is a small sample for a target that ranges from 0 to
127, and a draw that happens to contain easy integers can flatter the network.
Scoring all 255 removes that.

The baseline is the same one the report used: always answer the median stopping
time. It looks at no input bits at all.

In [ ]:
# Cell 16 - Cross-validate Collatz stopping time across all 255 integers

cv_collatz = cross_validate(collatz_x, collatz_y, COLLATZ_BITS, "Collatz")

collatz_median = int(np.median(collatz_actual))
collatz_constant = np.full(len(collatz_actual), collatz_median)

# Keep these for Cell 17, which puts all three problems on one scale
cv_collatz_error = float(np.abs(cv_collatz - collatz_actual).mean())
cv_collatz_baseline = float(np.abs(collatz_constant - collatz_actual).mean())

print_cv_scores(
    [
        (
            "the trained network",
            int((cv_collatz == collatz_actual).sum()),
            len(collatz_actual),
            cv_collatz_error,
            "",
        ),
        (
            "always answer " + str(collatz_median),
            int((collatz_constant == collatz_actual).sum()),
            len(collatz_actual),
            cv_collatz_baseline,
            "learns nothing",
        ),
    ]
)

---
### Reading all four together

The left panel overlays the four training curves. Every one of them falls, which
is the trap: **a network descending nicely on its training data tells you nothing
about whether it learned anything.** Collatz drives its training error down about
as well as population count does.

The right panel is the honest measure, and it puts all three problems on one
scale by asking the same question of each: how big is the network's error
compared with the error made by a baseline that does no learning at all? Below
100 percent means its answers land closer than doing nothing. At or above 100
percent means the training bought nothing.

Percent-correct would not work here. Hitting a stopping time in 0 to 127 exactly
is a much harder target than a population count in 0 to 8, so the same score
means different things in different columns. Comparing each network against its
own baseline removes that distortion.

All three numbers were computed by the cross-validation cell that followed each
problem, so nothing is retrained here. Each rests on every integer in its range
rather than on one draw of 16, which is what makes the comparison worth trusting.

The last column of the table names **what is being measured**, because the three
problems do not measure the same thing:

| Label | Problem | What the number means |
| --- | --- | --- |
| `counts off` | Population count | How far the predicted bit count sits from the true one, averaged over every integer. The target runs 0 to 8, so 0.24 means a typical answer misses by about a quarter of one count. |
| `steps off` | Collatz | How far the predicted stopping time sits from the true one, averaged, measured in Collatz steps. The target runs 0 to 127, so 29.69 means a typical answer misses by about 30 steps. |
| `balanced error` | Primality | Prime is a yes or no label, so there is no distance between a right and a wrong answer to average. This is the average of two failure rates instead: the fraction of primes missed, and the fraction of composites wrongly called prime. Answering "composite" every time scores 0.50. |

Those three units cannot be compared with each other. Being 0.24 counts off is
excellent; being 0.24 balanced error would be poor. That is what the **relative**
column is for: it divides the network's error by its own baseline's error, which
cancels the units and leaves one question asked identically of all three.

Read that column against 100 percent, which is the score of a network that
learned nothing:

- **Well under 100** means the network's answers land closer than the baseline's,
  so training bought something real. Population count reaches 22 percent, roughly
  a fifth of the error that answering a constant would have made.
- **A little under 100** means a marginal gain worth treating with suspicion.
  Primality reaches 78 percent, having shaved a little off the constant answer
  and not much more.
- **At or above 100** means the training bought nothing at all. Collatz reaches
  104 percent, which is worse than never training.

The dashed line across the bar chart marks that 100 percent mark, so a bar
reaching past it is a network that lost to doing nothing.

In [ ]:
# Cell 17 - Compare all four problems side by side

histories = [
    ("XOR", xor_history),
    ("Population count", pop_history),
    ("Primality", prime_history),
    ("Collatz", collatz_history),
]

# Score every problem the same way: the network's average error as a
# fraction of the error made by a baseline that does no learning. Below
# 100% means its answers land closer than doing nothing, above means they
# land further off. Percent-correct cannot be compared across these three,
# because hitting a stopping time in 0..127 exactly is a far harder target
# than a population count in 0..8
# Every number here was computed by the cross-validation cell that
# followed each problem, so nothing is retrained and nothing rests on a
# single draw of 16 integers
errors = {
    "Population count": (cv_pop_error, cv_pop_baseline, "counts off"),
    # Primality is a label rather than a magnitude, so its error is 1
    # minus balanced accuracy, and a constant answer sits at 0.50
    "Primality": (1 - cv_balanced, 0.5, "balanced error"),
    "Collatz": (cv_collatz_error, cv_collatz_baseline, "steps off"),
}

figure, (left, right) = plt.subplots(1, 2, figsize=(13, 4.5))

for name, history in histories:
    left.plot(history, label=name)
left.set_title("Every network trains well\nTraining Error vs. Epoch")
left.set_xlabel("Epoch")
left.set_ylabel("Mean absolute error")
left.set_yscale("log")
left.grid(which="both", alpha=0.4)
left.legend()

names = list(errors)
relative = [100 * errors[name][0] / errors[name][1] for name in names]

# Every bar is the same color. The dashed line at 100% is what separates a
# result that beat its baseline from one that did not
right.bar(names, relative, color="tab:blue")
right.axhline(
    100,
    color="black",
    linestyle="--",
    linewidth=1,
    label="no better than doing nothing",
)
right.set_title("Only one of them learned\nHeld-Out Error as a Percent of the Baseline")
right.set_ylabel("Percent of the baseline's error")
right.grid(axis="y", alpha=0.4)
right.legend()

plt.tight_layout()
plt.show()

table = Table(
    title="Held-out error against a baseline that does no learning",
    box=box.SIMPLE_HEAVY,
    title_justify="left",
)
table.add_column("problem")
table.add_column("network", justify="right")
table.add_column("baseline", justify="right")
table.add_column("relative", justify="right")
table.add_column("")

for name in names:
    network_error, baseline_error, units = errors[name]
    ratio = 100 * network_error / baseline_error
    table.add_row(
        name,
        f"{network_error:.2f}",
        f"{baseline_error:.2f}",
        f"{ratio:.0f}%",
        units,
    )

console.print(table)

---
### Where the knowledge actually lives

Everything each network learned is sitting in the `.npz` files, and it is nothing
but numbers. There is no rule written down anywhere, no lookup table, no comment
explaining what the network figured out. The knowledge is **distributed** across
several hundred weights, none of which means anything on its own.

The panels below show `weights_input_hidden1` from all three saved networks. That
matrix is the same shape in every case, 8 rows by 16 columns: one row per input
bit, one column per neuron in the first hidden layer. It is the layer the input
bits enter first, which makes it the most tempting place to go looking for
meaning.

Look at the three panels and try to say which is which. Nothing distinguishes
them. The one visible difference is that the Collatz weights spread wider, and
even that says nothing about what function was learned, only that the training
pushed harder to fit a target it could not fit.

The table underneath is the point. The only way to get knowledge out of a matrix
like this is to **pose a question**: feed in an input, run it forward, and read
what comes back. Those same 128 numbers answer "how many 1 bits" for one network
and "is this prime" for another, and no amount of staring at them reveals which.
This is why probing a trained model means querying it rather than reading it.

Each cell of that table shows the network's answer, then the true answer, and
marks a wrong one with `(*)`. Every integer listed comes from the held-out set,
so no network ever trained on any of them. Notice that being wrong is invisible
from the inside: the network returns a confident answer either way, and nothing
in the weights or the output says which answers to trust. Knowing that the
Collatz predictions are worthless took the whole cross-validation exercise, not
a look at the numbers.

In [ ]:
# Cell 18 - The learned weights as images, and the only way to read them

saved_networks = {
    "Population count": (LAB_DIR / "nn_popcount_weights.npz", 4),
    "Primality": (LAB_DIR / "nn_primes_weights.npz", 4),
    "Collatz": (LAB_DIR / "nn_collatz_weights.npz", COLLATZ_BITS),
}

figure, panels = plt.subplots(1, 3, figsize=(13, 3.6))
loaded = {}

for panel, (name, (path, output_bits)) in zip(panels, saved_networks.items()):
    network, _ = load_network(path)
    loaded[name] = network
    weights = network.weights_input_hidden1

    # Each panel is scaled to its own largest weight, so the three are
    # compared on pattern rather than on magnitude. Red is positive,
    # blue is negative, and the shade is how large
    limit = np.abs(weights).max()
    image = panel.imshow(weights, cmap="RdBu_r", vmin=-limit, vmax=limit, aspect="auto")
    panel.set_title(f"{name}\nweights_input_hidden1, range +/-{limit:.2f}", fontsize=10)
    panel.set_xlabel("hidden neuron")
    panel.set_ylabel("input bit")
    panel.set_yticks(range(8))
    figure.colorbar(image, ax=panel, fraction=0.046)

plt.tight_layout()
plt.show()

# Ask all three networks the same integers and watch them answer different
# questions from matrices that look identical
table = Table(
    title="The only way to read a network is to ask it something",
    box=box.SIMPLE_HEAVY,
    title_justify="left",
)
table.add_column("n", justify="right")
table.add_column("input bits")
table.add_column("population count")
table.add_column("prime?")
table.add_column("Collatz steps")


def aligned(pairs: list[tuple[str, str]]) -> list[str]:
    """Format "said / true" so a column lines up on its slash.

    Both halves are padded to the widest entry in that column, and the
    marker occupies the same width whether or not it is shown, so the
    slashes and the (*) each fall in one place down the whole column.
    """
    said_width = max(len(said) for said, _ in pairs)
    true_width = max(len(true) for _, true in pairs)
    return [
        f"{said:>{said_width}} / {true:>{true_width}}"
        + ("    " if said == true else " (*)")
        for said, true in pairs
    ]


# These are all integers from the held-out set, so no network ever trained
# on them, and the wrong answers are the same failures the reports measured
questions = [int(value) for value in HELD_OUT[:6]]
asked = []

for n in questions:
    row = x_8bit[[n]]
    pop_said = int(decode_bits(loaded["Population count"].forward(row), 4)[0])
    prime_said = bool(decode_bits(loaded["Primality"].forward(row), 4)[0] == 9)
    collatz_said = int(
        decode_bits(loaded["Collatz"].forward(collatz_x[[n - FIRST_N]]), COLLATZ_BITS)[
            0
        ]
    )
    asked.append(
        [
            (str(pop_said), str(pop_actual[n])),
            ("yes" if prime_said else "no", "yes" if prime_actual[n] else "no"),
            (str(collatz_said), str(collatz_actual[n - FIRST_N])),
        ]
    )

columns = [aligned([row[i] for row in asked]) for i in range(3)]

for index, n in enumerate(questions):
    table.add_row(str(n), format(n, "08b"), *[column[index] for column in columns])

console.print(table)
print("Each cell reads: what the network answered / the true answer.")
print("(*) marks an answer the network got wrong.")
print("Same input, same shaped matrices, three unrelated questions answered.")
print("None of it could be read off the weights directly.")